# Publications markdown generator for academicpages

Takes a TSV of publications with metadata and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). The core python code is also in `publications.py`. Run either from the `markdown_generator` folder after replacing `publications.tsv` with one containing your data.

TODO: Make this work with BibTex and other databases of citations, rather than Stuart's non-standard TSV format and citation style.


## Data format

The TSV needs to have the following columns: pub_date, title, venue, excerpt, citation, site_url, and paper_url, with a header at the top. 

- `excerpt` and `paper_url` can be blank, but the others must have values. 
- `pub_date` must be formatted as YYYY-MM-DD.
- `url_slug` will be the descriptive part of the .md file and the permalink URL for the page about the paper. The .md file will be `YYYY-MM-DD-[url_slug].md` and the permalink will be `https://[yourdomain]/publications/YYYY-MM-DD-[url_slug]`

This is how the raw file looks (it doesn't look pretty, use a spreadsheet or other program to edit and create).

In [9]:
!cat publications.tsv

pub_date	title	venue	excerpt	citation	url_slug	paper_url
2023-03-30	Assessing the replication landscape in experimental linguistics	Glossa Psycholinguistics	Replications are an integral part of cumulative experimental science. Yet many scientific disciplines do not replicate much because novel confirmatory findings are valued over direct replications. To provide a systematic assessment of the replication landscape in experimental linguistics, the present study estimated replication rates for over 50,000 articles across 98 journals. We used automatic string matching using the Web of Science combined with in-depth manual inspections of 274 papers. The median rate of mentioning the search string “replicat*” was as low as 1.7%. Subsequent manual analyses of articles containing the search string revealed that only 4% of these contained a direct replication, i.e., a study that aims to arrive at the same scientific conclusions as an initial study by using exactly the same methodology. Less th

## Import pandas

We are using the very handy pandas library for dataframes.

In [10]:
import pandas as pd

## Import TSV

Pandas makes this easy with the read_csv function. We are using a TSV, so we specify the separator as a tab, or `\t`.

I found it important to put this data in a tab-separated values format, because there are a lot of commas in this kind of data and comma-separated values can get messed up. However, you can modify the import statement, as pandas also has read_excel(), read_json(), and others.

In [11]:
publications = pd.read_csv("publications.tsv", sep="\t", header=0)
publications


,pub_date,title,venue,excerpt,citation,url_slug,paper_url
0,2023-03-30,Assessing the replication landscape in experim...,Glossa Psycholinguistics,Replications are an integral part of cumulativ...,"Kobrock, Kristina & Roettger, Timo B. (2023). ...",ReplicationLing,https://doi.org/10.5070/G6011135
1,2024-05-20,Context Shapes Emergent Communication about Co...,Proceedings of the 2024 Joint International Co...,We study the communication of concepts at diff...,"Kobrock, Kristina, Ohmer, Xenia, Bruni, Elia &...",ContextAware,https://aclanthology.org/2024.lrec-main.339.pdf
2,2024-07-25,Superordinate referring expressions in abstrac...,Proceedings of the Annual Meeting of the Cogni...,We study referential communication about conce...,"Kobrock, Kristina, Uhlemann, Charlotte & Gotzn...",ConceptRG,https://osf.io/cv74u
3,2024-11-20,"Feeling good, approaching the positive","Frontiers in Psychology, Section Cognitive Sci...",Approach and avoidance behaviors have been ext...,"Kobrock, Kristina, Solzbacher, Johannes, Gotzn...",Affect-AAT,https://doi.org/10.3389/fpsyg.2024.1491612
4,2024-10-28,Agents can generalize to novel levels of abstr...,Findings of the Association for Computational ...,We study abstraction in an emergent communicat...,"Kobrock, K., Ohmer, X., Bruni, E., & Gotzner, ...",Zero-shotAbstraction,https://doi.org/10.18653/v1/2025.findings-acl.455
5,2026-07-06,The Efficiency of Ambiguity: Abstract Referenc...,submitted to Cognitive Science,"We investigate how context granularity, i.e. w...","Briotto, Anna, Kobrock, Kristina & Bruni, Elia...",ContextGranularity,https://doi.org/10.31234/osf.io/3btvj_v2
6,2026-06-11,The role of pragmatic mechanisms in referentia...,PLOS Computational Biology,We model pragmatic mechanisms of referential c...,"Kobrock, K., Ohmer, X., Bruni, E., & Gotzner, ...",PragmaticMechanisms,https://journals.plos.org/ploscompbiol/article...


## Escape special characters

YAML is very picky about how it takes a valid string, so we are replacing single and double quotes (and ampersands) with their HTML encoded equivilents. This makes them look not so readable in raw format, but they are parsed and rendered nicely.

In [12]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    """Produce entities within text."""
    return "".join(html_escape_table.get(c,c) for c in text)

## Creating the markdown files

This is where the heavy lifting is done. This loops through all the rows in the TSV dataframe, then starts to concatentate a big string (```md```) that contains the markdown for each type. It does the YAML metadata first, then does the description for the individual page.

In [13]:
import os
for row, item in publications.iterrows():
    
    md_filename = str(item.pub_date) + "-" + item.url_slug + ".md"
    html_filename = str(item.pub_date) + "-" + item.url_slug
    year = item.pub_date[:4]
    
    ## YAML variables
    
    md = "---\ntitle: \""   + item.title + '"\n'
    
    md += """collection: publications"""
    
    md += """\npermalink: /publication/""" + html_filename
    
    if len(str(item.excerpt)) > 5:
        md += "\nexcerpt: '" + html_escape(item.excerpt) + "'"
    
    md += "\ndate: " + str(item.pub_date) 
    
    md += "\nvenue: '" + html_escape(item.venue) + "'"
    
    if len(str(item.paper_url)) > 5:
        md += "\npaperurl: '" + item.paper_url + "'"
    
    md += "\ncitation: '" + html_escape(item.citation) + "'"
    
    md += "\n---"
    
    ## Markdown description for individual page
        
    if len(str(item.excerpt)) > 5:
        md += "\n" + html_escape(item.excerpt) + "\n"
    
    if len(str(item.paper_url)) > 5:
        md += "\n[Download paper here](" + item.paper_url + ")\n" 
        
    md += "\nRecommended citation: " + item.citation
    
    md_filename = os.path.basename(md_filename)
       
    with open("../_publications/" + md_filename, 'w') as f:
        f.write(md)

These files are in the publications directory, one directory below where we're working from.

In [14]:
!ls ../_publications/

2023-03-30-ReplicationLing.md      2025-02-14-ContextGranularity.md
2024-05-20-ContextAware.md         2026-06-11-PragmaticMechanisms.md
2024-07-25-ConceptRG.md            2026-06-PragmaticMechanisms.md
2024-10-28-Zero-shotAbstraction.md 2026-07-06-ContextGranularity.md
2024-11-20-Affect-AAT.md           2026-07-ContextGranularity.md


In [15]:
!cat ../_publications/2009-10-01-paper-title-number-1.md

cat: ../_publications/2009-10-01-paper-title-number-1.md: No such file or directory
